# Partie 5 : Création du Data Warehouse (AWS RDS)

## Objectif
Structurer les données brutes dans une base relationnelle SQL hébergée dans le Cloud pour les rendre exploitables par requêtes.

## Choix techniques
- **AWS RDS (PostgreSQL)** : base de données managée, adaptée aux requêtes analytiques.
- **SQLAlchemy** : ORM pour insérer les DataFrames Pandas directement en base, avec gestion automatique des types.

## Processus ETL
- **Extract** : lecture des CSV depuis AWS S3 (Data Lake), pas en local.
- **Transform** : conversion des scores en numérique, gestion des valeurs manquantes.
- **Load** : insertion dans PostgreSQL via `to_sql()`.

## Note sur le `city_id`
Le champ `city_id` (identifiant unique par ville) est généré dans le notebook 01 et propagé dans tous les CSV. Il permet des jointures fiables entre les tables `weather` et `hotels`.

In [1]:
import io
import os

import boto3
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Chargement de la configuration
load_dotenv()

# AWS S3 (source : Data Lake)
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
BUCKET_NAME = os.getenv("AWS_BUCKET_NAME")

# AWS RDS (destination : Data Warehouse)
DB_ENDPOINT = os.getenv("DB_HOST")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_USER = os.getenv("DB_USER")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

# --- EXTRACT : Lecture depuis S3 ---
print("EXTRACT : Lecture des CSV depuis S3...")

session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name="eu-west-3"
)
s3 = session.client("s3")


def read_csv_from_s3(bucket, key):
    """Télécharge un CSV depuis S3 et retourne un DataFrame."""
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.BytesIO(obj['Body'].read()))


df_weather = read_csv_from_s3(BUCKET_NAME, 'cities_weather_data_7days.csv')
print(f"   weather : {len(df_weather)} lignes")

df_hotels = read_csv_from_s3(BUCKET_NAME, 'hotels_data.csv')
print(f"   hotels  : {len(df_hotels)} lignes")

# --- TRANSFORM : Nettoyage ---
print("\nTRANSFORM : Nettoyage des données...")

# Conversion du score en numérique
df_hotels['score'] = pd.to_numeric(df_hotels['score'], errors='coerce')
nb_missing = df_hotels['score'].isna().sum()
print(f"   Scores convertis en float ({nb_missing} valeurs manquantes)")

# --- LOAD : Insertion dans RDS ---
conn_str = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_ENDPOINT}:{DB_PORT}/{DB_NAME}"
)

try:
    print(f"\nLOAD : Connexion à RDS...")
    engine = create_engine(conn_str)

    with engine.connect() as connection:
        connection.execute(text("SELECT 1"))
        print("   Connexion réussie")

    # Insertion des tables
    df_weather.to_sql('weather', engine, if_exists='replace', index=False)
    print(f"   Table 'weather' : {len(df_weather)} lignes insérées.")

    df_hotels.to_sql('hotels', engine, if_exists='replace', index=False)
    print(f"   Table 'hotels'  : {len(df_hotels)} lignes insérées.")

    print("\nETL terminé : S3 -> Transform -> RDS PostgreSQL")

except Exception as e:
    print(f"\nErreur critique : {e}")

EXTRACT : Lecture des CSV depuis S3...
   weather : 35 lignes
   hotels  : 700 lignes

TRANSFORM : Nettoyage des données...
   Scores convertis en float (5 valeurs manquantes)

LOAD : Connexion à RDS...
   Connexion réussie
   Table 'weather' : 35 lignes insérées.
   Table 'hotels'  : 700 lignes insérées.

ETL terminé : S3 -> Transform -> RDS PostgreSQL
